# Kestrel — Colab Simulation Session

Colab runtimes are disposable. Run all **Initialization** cells (0–5) in order **every session**.

| Section | Cells | Description |
|---|---|---|
| ⚡ Initialization | 0–5 | Run every session · stop on [NO-GO] |
| 📊 Live Monitoring | M1–M4 | Re-run any time · M4 follows live stream |
| 🎛 Controls | C1–C3 | Daemon status · restart · graceful stop |
| 📈 Backtest | A–F | Independent · no live daemon required |

> `ENV=dev` · `TESTNET=true` · simulation engine only · **no real orders placed**
>
> See CLAUDE.md §29 for full protocol.

---
## 🧪 QUICK TEST — No API Keys Required

Just want to verify the signal engine works? Run **only these 3 cells**.
No exchange account, no database, no Telegram needed.
Uses public Binance OHLCV data (no auth) and dummy config values.

In [ ]:
# QT-1 — Setup: install deps + clone repo  (run once per Colab session)
import subprocess, pathlib, os

subprocess.run(
    ['pip', 'install', '-q', 'ccxt', 'matplotlib', 'numpy', 'python-dotenv'],
    check=True, capture_output=True,
)
print('OK  Python packages')

REPO_URL    = 'https://github.com/azzindani/kestrel.git'
KESTREL_DIR = pathlib.Path('/content/kestrel')

if (KESTREL_DIR / '.git').exists():
    r = subprocess.run(['git', 'pull', '--ff-only'], cwd=KESTREL_DIR,
                       capture_output=True, text=True)
    print(f'OK  Repo: {r.stdout.strip() or "already up to date"}')
else:
    r = subprocess.run(['git', 'clone', REPO_URL, str(KESTREL_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('Clone failed: ' + r.stderr)
    print(f'OK  Repo cloned -> {KESTREL_DIR}')

os.chdir(KESTREL_DIR)
import sys; sys.path.insert(0, str(KESTREL_DIR))
print(f'OK  Ready')

In [ ]:
# QT-2 — Fetch 90 days of BTC/USDT 5m candles + build indicators
# No API key needed — public Binance endpoint.
import ccxt, time, os
from collections import deque
from src.config import Candle, compute_candle_geometry, load_params
from src.signal.indicators import compute_all_indicators

PAIR      = 'BTC/USDT'
TIMEFRAME = '5m'
DAYS      = 90

exchange = ccxt.binance({'enableRateLimit': True})
since    = int((time.time() - DAYS * 86400) * 1000)
ohlcvs   = []

print(f'Fetching {DAYS}d of {PAIR} {TIMEFRAME}...')
while True:
    batch = exchange.fetch_ohlcv(PAIR, TIMEFRAME, since=since, limit=1000)
    if not batch:
        break
    ohlcvs.extend(batch)
    since = batch[-1][0] + 1
    if len(batch) < 1000:
        break
    print(f'  {len(ohlcvs):,} candles...', end='\r')

print(f'OK  {len(ohlcvs):,} candles fetched')

params  = load_params('params.json')
BOT_ID  = 'quicktest'
buf     = deque(maxlen=120)
candles = []

for i, row in enumerate(ohlcvs):
    ts, o, h, l, c, v = row
    geom = compute_candle_geometry(o, h, l, c)
    raw  = Candle(bot_id=BOT_ID, ts=ts, pair='BTCUSDT', timeframe='5m',
                  open=o, high=h, low=l, close=c, volume=v, **geom)
    buf.append(raw)
    inds = compute_all_indicators(list(buf), ema_fast=params.ema_fast,
                                  ema_slow=params.ema_slow)
    full = Candle(bot_id=BOT_ID, ts=ts, pair='BTCUSDT', timeframe='5m',
                  open=o, high=h, low=l, close=c, volume=v, **geom, **inds)
    buf[-1] = full
    candles.append(full)
    if (i + 1) % 5000 == 0:
        print(f'  Built {i+1:,} / {len(ohlcvs):,}...', end='\r')

print(f'OK  {len(candles):,} candles with indicators')

# Regime summary
recent = candles[-500:]
n      = len(recent)
for reg in ('TRENDING', 'VOLATILE', 'RANGING', 'QUIET'):
    cnt = sum(1 for c in recent if c.regime == reg)
    print(f'  {reg:10}  {cnt:4d}  ({cnt/n*100:.0f}%)')

In [ ]:
# QT-3 — Walk-forward backtest + equity curve
# Dummy config values — nothing connects to DB, exchange, or Telegram.
import os, matplotlib.pyplot as plt
from src.config import AppConfig
from src.backtest.runner import walk_forward, run_backtest

os.environ.update({
    'ENV': 'dev',
    'BOT_ID': 'quicktest',
    'EXCHANGE': 'binance',
    'API_KEY': 'dummy',
    'API_SECRET': 'dummy',
    'TESTNET': 'true',
    'DB_HOST': 'localhost',
    'DB_PORT': '5432',
    'DB_NAME': 'kestrel',
    'DB_USER': 'kestrel',
    'DB_PASSWORD': 'dummy',
    'PAIR': 'BTCUSDT',
    'TIMEFRAME_ENTRY': '5m',
    'TIMEFRAME_REGIME': '15m',
    'LEVERAGE': '20',
    'BUCKET_SIZE_USDT': '10.0',
    'MAX_ACTIVE_BUCKETS': '1',
    'TELEGRAM_TOKEN': 'dummy',
    'TELEGRAM_CHAT_ID': '0',
    'LOG_LEVEL': 'DEBUG',
})

cfg     = AppConfig.from_mapping(os.environ)
results = walk_forward(candles, params, cfg)
oos     = results['out_sample']

# ── Verdict ──────────────────────────────────────────────────────────────
win_rate = oos.get('win_rate', 0)
avg_pnl  = oos.get('avg_pnl_usdt', 0)
drawdown = oos.get('max_drawdown_pct', 100)
sharpe   = oos.get('sharpe_ratio', 0)
total    = oos.get('total_trades', 0)
reasons  = oos.get('close_reasons', {})

checks = [
    ('Win rate > 55%',         win_rate >= 0.55,  f'{win_rate*100:.1f}%'),
    ('Avg PnL/trade > $0.018', avg_pnl  >= 0.018, f'${avg_pnl:.4f}'),
    ('Max drawdown < 30%',     drawdown <= 30.0,  f'{drawdown:.1f}%'),
    ('Sharpe ratio > 0',       sharpe   >  0,     f'{sharpe:.2f}'),
    ('Trades >= 20',           total    >= 20,    f'{total}'),
]

print('── Out-of-Sample Results (no API keys used) ───────────────')
all_pass = True
for label, passed, val in checks:
    sym      = 'PASS' if passed else 'FAIL'
    all_pass = all_pass and passed
    print(f'  [{sym}]  {label:<30}  {val}')
print(f'  Close reasons: {reasons}')
verdict = '[GO]' if all_pass else '[NO-GO — review params]'
print(f'\n  {verdict}')
print('──────────────────────────────────────────────────────────')

# ── Equity curve ────────────────────────────────────────────────────────
split       = int(len(candles) * 0.6)
test_result = run_backtest(candles[split:], params, cfg)
equity      = test_result['equity_curve']

peak = 0.0
dds  = [((peak := max(peak, e)), e - peak)[1] for e in equity]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6),
                                gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(equity, lw=1.2, color='steelblue')
ax1.axhline(0, color='gray', ls='--', lw=0.8)
ax1.fill_between(range(len(equity)), equity, 0,
                 where=[e >= 0 for e in equity], alpha=0.15, color='green')
ax1.fill_between(range(len(equity)), equity, 0,
                 where=[e <  0 for e in equity], alpha=0.15, color='red')
ax1.set_title('Out-of-Sample Equity Curve  (40% test window — no API keys)', fontsize=12)
ax1.set_ylabel('Cumulative PnL (USDT)')
ax1.grid(True, alpha=0.3)
ax2.fill_between(range(len(dds)), dds, 0, color='red', alpha=0.4)
ax2.set_ylabel('Drawdown (USDT)')
ax2.set_xlabel('Candle index (5-min bars)')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Peak: ${max(equity) if equity else 0:.4f}  |  '
      f'Final: ${equity[-1] if equity else 0:.4f}  |  '
      f'Max DD: ${min(dds) if dds else 0:.4f}')

---
## ⚡ INITIALIZATION — Run Every Session

In [ ]:
# Cell 0 — Kill existing Kestrel processes + clean state
# Run this first every session. Safe to re-run at any time.
import os, signal, pathlib, subprocess, time

KESTREL_DIR = pathlib.Path('/content/kestrel')

for pattern in ('src.engine.daemon', 'src.engine.watchdog'):
    r = subprocess.run(['pgrep', '-f', pattern], capture_output=True, text=True)
    for pid_str in r.stdout.strip().splitlines():
        pid_str = pid_str.strip()
        if not pid_str:
            continue
        try:
            os.kill(int(pid_str), signal.SIGTERM)
            print(f'  SIGTERM -> {pattern}  pid={pid_str}')
        except ProcessLookupError:
            pass

time.sleep(2)

pid_file = KESTREL_DIR / 'kestrel.pid'
if pid_file.exists():
    pid_file.unlink()
    print('  Removed stale kestrel.pid')

print('OK  Environment clean')

In [ ]:
# Cell 1 — System packages + clone / update repo
import subprocess, pathlib, os

subprocess.run(
    ['apt-get', 'install', '-y', '-q', 'postgresql-client'],
    check=True, capture_output=True,
)
print('OK  postgresql-client')

subprocess.run(
    ['pip', 'install', '-q', 'asyncpg', 'python-dotenv', 'ccxt', 'matplotlib', 'numpy'],
    check=True, capture_output=True,
)
print('OK  Python packages  (asyncpg, ccxt, matplotlib)')

REPO_URL    = 'https://github.com/azzindani/kestrel.git'
KESTREL_DIR = pathlib.Path('/content/kestrel')

if (KESTREL_DIR / '.git').exists():
    r = subprocess.run(
        ['git', 'pull', '--ff-only'],
        cwd=KESTREL_DIR, capture_output=True, text=True,
    )
    msg = r.stdout.strip() or 'already up to date'
    print(f'OK  Repo: {msg}')
else:
    r = subprocess.run(
        ['git', 'clone', REPO_URL, str(KESTREL_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError('Clone failed — check REPO_URL and credentials.' + r.stderr)
    print(f'OK  Repo cloned -> {KESTREL_DIR}')

os.chdir(KESTREL_DIR)
print(f'OK  cwd={os.getcwd()}')

In [ ]:
# Cell 2 — Write .env from Colab Secrets
# Set these keys in Colab > key icon > Secrets before running:
#   BOT_ID  EXCHANGE  API_KEY  API_SECRET
#   DB_HOST  DB_PORT  DB_NAME  DB_USER  DB_PASSWORD
#   PAIR  TELEGRAM_TOKEN  TELEGRAM_CHAT_ID
from google.colab import userdata
import pathlib

def _s(key, default=''):
    try:
        v = userdata.get(key)
        return v if v else default
    except Exception:
        return default

env_text = '\n'.join([
    'ENV=dev',
    'BOT_ID='           + _s('BOT_ID'),
    'EXCHANGE='         + _s('EXCHANGE'),
    'API_KEY='          + _s('API_KEY'),
    'API_SECRET='       + _s('API_SECRET'),
    'TESTNET=true',
    'DB_HOST='          + _s('DB_HOST'),
    'DB_PORT='          + _s('DB_PORT', '5432'),
    'DB_NAME='          + _s('DB_NAME'),
    'DB_USER='          + _s('DB_USER'),
    'DB_PASSWORD='      + _s('DB_PASSWORD'),
    'PAIR='             + _s('PAIR', 'BTCUSDT'),
    'TIMEFRAME_ENTRY=5m',
    'TIMEFRAME_REGIME=15m',
    'LEVERAGE=20',
    'BUCKET_SIZE_USDT=10.0',
    'MAX_ACTIVE_BUCKETS=1',
    'TELEGRAM_TOKEN='   + _s('TELEGRAM_TOKEN'),
    'TELEGRAM_CHAT_ID=' + _s('TELEGRAM_CHAT_ID'),
    'LOG_LEVEL=DEBUG',
]) + '\n'

env_path = pathlib.Path('/content/kestrel/.env')
env_path.write_text(env_text)
print('OK  .env written  (20 keys)')
print('    ENV=dev  TESTNET=true  LEVERAGE=20  BUCKET_SIZE_USDT=10.0')

In [ ]:
# Cell 3 — Install + validate  (must print [GO] — abort if [NO-GO])
import subprocess, os

os.chdir('/content/kestrel')

proc = subprocess.Popen(
    ['bash', 'scripts/install.sh'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode != 0:
    raise SystemExit('[ABORT] install.sh failed — fix the issue above before running Cell 4')

In [ ]:
# Cell 4 — Start daemon  (ENV=dev · simulation engine · TESTNET=true · no real orders)
import subprocess, time, pathlib, os, signal

KESTREL_DIR = pathlib.Path('/content/kestrel')
VENV_PY     = KESTREL_DIR / 'venv/bin/python'
DAEMON_LOG  = KESTREL_DIR / 'daemon.log'
PID_FILE    = KESTREL_DIR / 'kestrel.pid'

os.chdir(KESTREL_DIR)

if PID_FILE.exists():
    pid = int(PID_FILE.read_text().strip())
    try:
        os.kill(pid, 0)
        print(f'Daemon already running  pid={pid}')
        print('Run Cell C2 to restart, or Cell 0 to kill first.')
        raise SystemExit('already running')
    except ProcessLookupError:
        PID_FILE.unlink()

with open(DAEMON_LOG, 'w') as fh:
    proc = subprocess.Popen(
        [str(VENV_PY), '-m', 'src.engine.daemon'],
        stdout=fh,
        stderr=subprocess.STDOUT,
        cwd=str(KESTREL_DIR),
    )

PID_FILE.write_text(str(proc.pid))
print(f'Daemon starting  pid={proc.pid}')
print(f'Log: {DAEMON_LOG}')

for i in range(15):
    time.sleep(1)
    if proc.poll() is not None:
        print(f'\nDaemon exited early  code={proc.returncode}')
        lines = DAEMON_LOG.read_text().splitlines()[-60:]
        print('\n'.join(lines))
        raise SystemExit('Daemon failed to start — see log above')
    print(f'  [{i+1:2d}s] starting...', end='\r')

print(f'\nOK  Daemon running  pid={proc.pid}')
print('    ENV=dev  TESTNET=true  simulation only — no real orders')

In [ ]:
# Cell 5 — Verify startup: heartbeat + recent events from DB
import asyncio, os, datetime
import asyncpg
from dotenv import load_dotenv

load_dotenv('/content/kestrel/.env')

async def _verify():
    u   = os.environ['DB_USER']
    pw  = os.environ['DB_PASSWORD']
    h   = os.environ['DB_HOST']
    p   = os.environ['DB_PORT']
    db  = os.environ['DB_NAME']
    bot = os.environ['BOT_ID']
    dsn = f'postgresql://{u}:{pw}@{h}:{p}/{db}'

    try:
        conn = await asyncio.wait_for(asyncpg.connect(dsn=dsn), timeout=10)
    except Exception as exc:
        print(f'Cannot connect to DB: {exc}')
        return

    hb = await conn.fetchrow(
        'SELECT ts, status, pid FROM heartbeats WHERE bot_id=$1', bot
    )
    if hb:
        age = (datetime.datetime.utcnow().timestamp() * 1000 - hb['ts']) / 1000
        sym = 'OK  ' if age < 90 else 'STALE'
        status = hb['status']
        pid    = hb['pid']
        print(f'{sym}  Heartbeat  status={status}  pid={pid}  age={age:.0f}s')
    else:
        print('WAIT  No heartbeat yet — daemon may still be starting up')

    rows = await conn.fetch(
        'SELECT ts, level, category, message FROM events '
        'WHERE bot_id=$1 ORDER BY ts DESC LIMIT 15',
        bot,
    )
    if rows:
        print(f'\nStartup events ({len(rows)} most recent):')
        for r in reversed(rows):
            ts  = datetime.datetime.utcfromtimestamp(r['ts'] / 1000).strftime('%H:%M:%S')
            lvl = r['level']
            cat = r['category']
            msg = r['message']
            print(f'  [{ts}] {lvl:8}  {cat:10}  {msg}')
    else:
        print('\nNo events in DB yet — daemon may still be connecting to exchange')

    await conn.close()

asyncio.run(_verify())

---
## 📊 LIVE MONITORING

All cells below are re-runnable at any time without restarting the daemon.

In [ ]:
# Cell M1 — Recent events  (re-run to refresh)
import asyncio, os, datetime
import asyncpg
from dotenv import load_dotenv

load_dotenv('/content/kestrel/.env')

async def _events(limit=30):
    u, pw, h, p, db = (
        os.environ['DB_USER'], os.environ['DB_PASSWORD'],
        os.environ['DB_HOST'], os.environ['DB_PORT'], os.environ['DB_NAME'],
    )
    bot = os.environ['BOT_ID']
    dsn = f'postgresql://{u}:{pw}@{h}:{p}/{db}'
    conn = await asyncpg.connect(dsn=dsn, command_timeout=10)

    rows = await conn.fetch(
        'SELECT ts, level, category, message FROM events '
        'WHERE bot_id=$1 ORDER BY ts DESC LIMIT $2',
        bot, limit,
    )
    await conn.close()

    print(f'{"TIME":8}  {"LEVEL":8}  {"CATEGORY":10}  MESSAGE')
    print('-' * 72)
    for r in reversed(rows):
        ts  = datetime.datetime.utcfromtimestamp(r['ts'] / 1000).strftime('%H:%M:%S')
        lvl = r['level']
        cat = r['category']
        msg = r['message']
        print(f'{ts}  {lvl:8}  {cat:10}  {msg}')

asyncio.run(_events())

In [ ]:
# Cell M2 — Session statistics  (re-run to refresh)
import asyncio, os
import asyncpg
from dotenv import load_dotenv

load_dotenv('/content/kestrel/.env')

async def _stats():
    u, pw, h, p, db = (
        os.environ['DB_USER'], os.environ['DB_PASSWORD'],
        os.environ['DB_HOST'], os.environ['DB_PORT'], os.environ['DB_NAME'],
    )
    bot = os.environ['BOT_ID']
    dsn = f'postgresql://{u}:{pw}@{h}:{p}/{db}'
    conn = await asyncpg.connect(dsn=dsn, command_timeout=10)

    t = await conn.fetchrow('''
        SELECT
            COUNT(*)                                              AS total,
            COUNT(*) FILTER (WHERE exit_ts IS NOT NULL)          AS closed,
            COUNT(*) FILTER (WHERE exit_ts IS NULL)              AS open,
            COUNT(*) FILTER (WHERE pnl_net_usdt > 0)             AS wins,
            COUNT(*) FILTER (WHERE pnl_net_usdt <= 0
                               AND exit_ts IS NOT NULL)          AS losses,
            COALESCE(SUM(pnl_net_usdt)
                FILTER (WHERE exit_ts IS NOT NULL), 0)           AS total_pnl,
            COALESCE(AVG(pnl_pct)
                FILTER (WHERE exit_ts IS NOT NULL
                          AND pnl_net_usdt > 0),  0)             AS avg_win_pct,
            COALESCE(AVG(pnl_pct)
                FILTER (WHERE exit_ts IS NOT NULL
                          AND pnl_net_usdt <= 0), 0)             AS avg_loss_pct
        FROM trades WHERE bot_id=$1 AND env='dev'
    ''', bot)

    s = await conn.fetchrow('''
        SELECT
            COUNT(*)                                        AS total,
            COUNT(*) FILTER (WHERE outcome='fired')         AS fired,
            COUNT(*) FILTER (WHERE outcome='rejected')      AS rejected
        FROM signals WHERE bot_id=$1
    ''', bot)

    cr = await conn.fetch('''
        SELECT close_reason, COUNT(*) AS cnt
        FROM trades WHERE bot_id=$1 AND env='dev' AND exit_ts IS NOT NULL
        GROUP BY close_reason ORDER BY cnt DESC
    ''', bot)

    await conn.close()

    closed   = t['closed'] or 0
    wins     = t['wins']   or 0
    win_rate = wins / closed * 100 if closed > 0 else 0.0
    pnl      = float(t['total_pnl'])
    aw       = float(t['avg_win_pct'])
    al       = float(t['avg_loss_pct'])

    print('── Session Statistics ─────────────────────────────────────')
    print(f'  Trades      {t["total"]:4d}  (closed={closed}  open={t["open"]})')
    print(f'  Win / Loss  {wins} W  {t["losses"]} L  ({win_rate:.1f}% win rate)')
    print(f'  Total PnL   ${pnl:+.4f} USDT')
    print(f'  Avg win     {aw:+.2f}%   Avg loss  {al:+.2f}%')
    print(f'  Signals     {s["total"]} total  {s["fired"]} fired  {s["rejected"]} rejected')
    reasons = {r["close_reason"]: r["cnt"] for r in cr}
    print(f'  Close reasons: {reasons}')
    print('───────────────────────────────────────────────────────────')

asyncio.run(_stats())

In [ ]:
# Cell M3 — Open positions  (re-run to refresh)
import asyncio, os, datetime
import asyncpg
from dotenv import load_dotenv

load_dotenv('/content/kestrel/.env')

async def _positions():
    u, pw, h, p, db = (
        os.environ['DB_USER'], os.environ['DB_PASSWORD'],
        os.environ['DB_HOST'], os.environ['DB_PORT'], os.environ['DB_NAME'],
    )
    bot = os.environ['BOT_ID']
    dsn = f'postgresql://{u}:{pw}@{h}:{p}/{db}'
    conn = await asyncpg.connect(dsn=dsn, command_timeout=10)

    rows = await conn.fetch('''
        SELECT pair, direction, pattern, leverage,
               entry_price, tp_price, sl_price, liquidation_price,
               size_usdt, entry_ts
        FROM trades
        WHERE bot_id=$1 AND env='dev' AND exit_ts IS NULL
        ORDER BY entry_ts DESC
    ''', bot)
    await conn.close()

    if not rows:
        print('No open positions')
        return

    now_ms = datetime.datetime.utcnow().timestamp() * 1000
    print(f'Open positions: {len(rows)}')
    print(f'  {"PAIR":10} {"DIR":5} {"PATTERN":25} {"LEV":4}'
          f' {"ENTRY":10} {"TP":10} {"SL":10} {"LIQ":10} {"SIZE":6} OPEN')
    print('  ' + '-' * 106)
    for r in rows:
        age_m = (now_ms - r['entry_ts']) / 60_000
        pair  = r['pair']
        dirn  = r['direction']
        patt  = r['pattern']
        lev   = r['leverage']
        ep    = float(r['entry_price'])
        tp    = float(r['tp_price'])
        sl    = float(r['sl_price'])
        liq   = float(r['liquidation_price'])
        sz    = float(r['size_usdt'])
        print(f'  {pair:10} {dirn:5} {patt:25} {lev:3}x'
              f' {ep:10.2f} {tp:10.2f} {sl:10.2f} {liq:10.2f} ${sz:<5.0f} {age_m:.0f}m')

asyncio.run(_positions())

In [ ]:
# Cell M4 — Follow live event stream  (interrupt kernel to stop)
import asyncio, os, datetime
import asyncpg
from dotenv import load_dotenv

load_dotenv('/content/kestrel/.env')

async def _follow():
    u, pw, h, p, db = (
        os.environ['DB_USER'], os.environ['DB_PASSWORD'],
        os.environ['DB_HOST'], os.environ['DB_PORT'], os.environ['DB_NAME'],
    )
    bot = os.environ['BOT_ID']
    dsn = f'postgresql://{u}:{pw}@{h}:{p}/{db}'
    conn = await asyncpg.connect(dsn=dsn)

    row = await conn.fetchrow(
        'SELECT COALESCE(MAX(id), 0) AS max_id FROM events WHERE bot_id=$1', bot
    )
    last_id = row['max_id']
    print(f'Following events  bot_id={bot}  (from id>{last_id})')
    print(f'{"TIME":8}  {"LEVEL":8}  {"CATEGORY":10}  MESSAGE')
    print('-' * 70)

    try:
        while True:
            new_rows = await conn.fetch(
                'SELECT id, ts, level, category, message '
                'FROM events WHERE bot_id=$1 AND id>$2 ORDER BY id ASC',
                bot, last_id,
            )
            for r in new_rows:
                ts  = datetime.datetime.utcfromtimestamp(r['ts'] / 1000).strftime('%H:%M:%S')
                lvl = r['level']
                cat = r['category']
                msg = r['message']
                print(f'{ts}  {lvl:8}  {cat:10}  {msg}', flush=True)
                last_id = r['id']
            await asyncio.sleep(2)
    except (KeyboardInterrupt, asyncio.CancelledError):
        print('\nStream stopped.')
    finally:
        await conn.close()

asyncio.run(_follow())

---
## 🎛 SIMULATION CONTROLS

In [ ]:
# Cell C1 — Daemon status check
import os, pathlib, asyncio, datetime
import asyncpg
from dotenv import load_dotenv

KESTREL_DIR = pathlib.Path('/content/kestrel')
load_dotenv(KESTREL_DIR / '.env')

# Process check
pid_file = KESTREL_DIR / 'kestrel.pid'
if pid_file.exists():
    pid = int(pid_file.read_text().strip())
    try:
        os.kill(pid, 0)
        print(f'OK   Process running  pid={pid}')
    except ProcessLookupError:
        print(f'FAIL PID file exists but process {pid} is not running')
else:
    print('WARN No PID file — daemon not running')

# Heartbeat check
async def _hb():
    u, pw, h, p, db = (
        os.environ['DB_USER'], os.environ['DB_PASSWORD'],
        os.environ['DB_HOST'], os.environ['DB_PORT'], os.environ['DB_NAME'],
    )
    bot = os.environ['BOT_ID']
    dsn = f'postgresql://{u}:{pw}@{h}:{p}/{db}'
    try:
        conn = await asyncio.wait_for(asyncpg.connect(dsn=dsn), timeout=5)
    except Exception as exc:
        print(f'WARN DB unreachable: {exc}')
        return
    row = await conn.fetchrow(
        'SELECT ts, status FROM heartbeats WHERE bot_id=$1', bot
    )
    await conn.close()
    if row:
        age = (datetime.datetime.utcnow().timestamp() * 1000 - row['ts']) / 1000
        sym = 'OK  ' if age < 90 else 'STALE'
        status = row['status']
        print(f'{sym} Heartbeat  status={status}  age={age:.0f}s')
    else:
        print('WARN No heartbeat in DB')

asyncio.run(_hb())

# Tail daemon log
log = KESTREL_DIR / 'daemon.log'
if log.exists():
    lines = log.read_text().splitlines()[-10:]
    print(f'\nLast {len(lines)} lines of daemon.log:')
    for ln in lines:
        print(f'  {ln}')

In [ ]:
# Cell C2 — Restart daemon  (graceful stop -> wait -> fresh start)
import os, signal, time, subprocess, pathlib

KESTREL_DIR = pathlib.Path('/content/kestrel')
VENV_PY     = KESTREL_DIR / 'venv/bin/python'
DAEMON_LOG  = KESTREL_DIR / 'daemon.log'
PID_FILE    = KESTREL_DIR / 'kestrel.pid'

# Stop existing
if PID_FILE.exists():
    pid = int(PID_FILE.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f'SIGTERM -> pid={pid}  (waiting for clean exit)...')
    except ProcessLookupError:
        pass

    for i in range(20):
        time.sleep(1)
        try:
            os.kill(pid, 0)
        except ProcessLookupError:
            print(f'Stopped cleanly after {i+1}s')
            break
    else:
        try:
            os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        print('Force-killed (SIGKILL)')

    PID_FILE.unlink(missing_ok=True)
else:
    print('No existing daemon — starting fresh')

time.sleep(2)

# Start fresh
with open(DAEMON_LOG, 'w') as fh:
    proc = subprocess.Popen(
        [str(VENV_PY), '-m', 'src.engine.daemon'],
        stdout=fh, stderr=subprocess.STDOUT,
        cwd=str(KESTREL_DIR),
    )

PID_FILE.write_text(str(proc.pid))
print(f'Daemon restarting  pid={proc.pid}')

for i in range(12):
    time.sleep(1)
    if proc.poll() is not None:
        print(f'\nDaemon exited early  code={proc.returncode}')
        print(DAEMON_LOG.read_text()[-2000:])
        raise SystemExit('Restart failed')
    print(f'  [{i+1:2d}s]...', end='\r')

print(f'\nOK  Daemon running  pid={proc.pid}')

In [ ]:
# Cell C3 — Graceful stop  (SIGTERM -> waits for positions to close -> exit)
import os, signal, time, pathlib

KESTREL_DIR = pathlib.Path('/content/kestrel')
PID_FILE    = KESTREL_DIR / 'kestrel.pid'

if not PID_FILE.exists():
    print('Daemon not running (no PID file)')
else:
    pid = int(PID_FILE.read_text().strip())
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        print(f'Process {pid} already gone — cleaning up')
        PID_FILE.unlink()
    else:
        print(f'SIGTERM -> pid={pid}  (daemon closing positions before exit)...')
        os.kill(pid, signal.SIGTERM)

        for i in range(30):
            time.sleep(1)
            try:
                os.kill(pid, 0)
                print(f'  [{i+1:2d}s] shutting down...', end='\r')
            except ProcessLookupError:
                PID_FILE.unlink(missing_ok=True)
                print(f'\nOK  Daemon stopped cleanly after {i+1}s')
                break
        else:
            print('\nWARN Daemon still alive after 30s — sending SIGKILL')
            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            PID_FILE.unlink(missing_ok=True)
            print('WARN Force-killed (positions may not have closed cleanly)')

---
## 📈 BACKTEST SESSION

Independent of the live daemon — runs on historical OHLCV fetched from Binance (no API key required).
Walk-forward protocol: **60% train / 40% test** per CLAUDE.md §30.
All cells use the same fee + slippage model as the live simulation engine.

In [ ]:
# Cell A — Fetch historical OHLCV  (90 days · no auth required)
import ccxt, time

PAIR      = 'BTC/USDT'
TIMEFRAME = '5m'
DAYS      = 90

exchange = ccxt.binance({'enableRateLimit': True})
since    = int((time.time() - DAYS * 86400) * 1000)

ohlcvs = []
print(f'Fetching {DAYS}d of {PAIR} {TIMEFRAME} candles...')
while True:
    batch = exchange.fetch_ohlcv(PAIR, TIMEFRAME, since=since, limit=1000)
    if not batch:
        break
    ohlcvs.extend(batch)
    since = batch[-1][0] + 1
    if len(batch) < 1000:
        break
    print(f'  {len(ohlcvs):,} candles...', end='\r')

days_actual = len(ohlcvs) * 5 / 60 / 24
print(f'OK  {len(ohlcvs):,} candles  ({days_actual:.1f} days of 5m bars)')

In [ ]:
# Cell B — Build Candle objects + compute indicators
import sys, os
sys.path.insert(0, '/content/kestrel')
os.chdir('/content/kestrel')

from collections import deque
from src.config import Candle, compute_candle_geometry, load_params
from src.signal.indicators import compute_all_indicators

params = load_params('params.json')
BOT_ID = 'backtest-colab'
buf    = deque(maxlen=120)
candles = []

for i, row in enumerate(ohlcvs):
    ts, o, h, l, c, v = row
    geom = compute_candle_geometry(o, h, l, c)
    raw  = Candle(bot_id=BOT_ID, ts=ts, pair='BTCUSDT', timeframe='5m',
                  open=o, high=h, low=l, close=c, volume=v, **geom)
    buf.append(raw)
    inds = compute_all_indicators(
        list(buf), ema_fast=params.ema_fast, ema_slow=params.ema_slow
    )
    full = Candle(bot_id=BOT_ID, ts=ts, pair='BTCUSDT', timeframe='5m',
                  open=o, high=h, low=l, close=c, volume=v, **geom, **inds)
    buf[-1] = full
    candles.append(full)
    if (i + 1) % 5000 == 0:
        print(f'  Built {i+1:,} / {len(ohlcvs):,}...', end='\r')

print(f'OK  {len(candles):,} candles with indicators')

In [ ]:
# Cell C — Indicator unit tests  (sanity check values before backtest)
from src.signal.indicators import compute_ema, compute_rsi, compute_atr

closes = [c.close for c in candles[-50:]]
ema9   = compute_ema(closes, 9)
ema21  = compute_ema(closes, 21)
rsi    = compute_rsi(closes, 14)
atr    = compute_atr(candles[-20:], 14)

assert 0 < rsi < 100, f'RSI out of range: {rsi}'
assert atr > 0,       f'ATR must be positive: {atr}'
assert ema9 is not None and ema21 is not None, 'EMA computation failed'
print(f'OK  EMA9={ema9:.2f}  EMA21={ema21:.2f}  RSI14={rsi:.1f}  ATR14={atr:.2f}')

# Regime distribution over last 500 candles
recent  = candles[-500:]
n       = len(recent)
regimes = ['TRENDING', 'VOLATILE', 'RANGING', 'QUIET']
for reg in regimes:
    cnt = sum(1 for c in recent if c.regime == reg)
    print(f'  {reg:10}  {cnt:4d} candles  ({cnt/n*100:.0f}%)')

In [ ]:
# Cell D — Walk-forward backtest  (60% train / 40% test)
import os
from dotenv import load_dotenv
load_dotenv('/content/kestrel/.env')

from src.config import AppConfig
from src.backtest.runner import walk_forward

cfg     = AppConfig.from_mapping(os.environ)
results = walk_forward(candles, params, cfg)

print('IN-SAMPLE  (train — 60%):')
for k, v in results['in_sample'].items():
    print(f'  {k:<32} {v}')

print()
print('OUT-OF-SAMPLE  (test — 40%  ← decision metric):')
for k, v in results['out_sample'].items():
    print(f'  {k:<32} {v}')

In [ ]:
# Cell E — Out-of-sample metrics + go/no-go verdict
oos = results['out_sample']

win_rate  = oos.get('win_rate', 0)
sharpe    = oos.get('sharpe_ratio', 0)
drawdown  = oos.get('max_drawdown_pct', 100)
avg_pnl   = oos.get('avg_pnl_usdt', 0)
total_tr  = oos.get('total_trades', 0)
reasons   = oos.get('close_reasons', {})

checks = [
    ('Win rate > 55%',           win_rate >= 0.55,  f'{win_rate*100:.1f}%'),
    ('Avg PnL/trade > $0.018',   avg_pnl  >= 0.018, f'${avg_pnl:.4f}'),
    ('Max drawdown < 30%',       drawdown <= 30.0,  f'{drawdown:.1f}%'),
    ('Sharpe ratio > 0',         sharpe   >  0,     f'{sharpe:.2f}'),
    ('Total trades >= 20',       total_tr >= 20,    f'{total_tr}'),
]

print('── Out-of-Sample Verdict ─────────────────────────────────────')
all_pass = True
for label, passed, val in checks:
    sym      = 'PASS' if passed else 'FAIL'
    all_pass = all_pass and passed
    print(f'  [{sym}]  {label:<30}  {val}')

print(f'  Close reasons: {reasons}')
print('──────────────────────────────────────────────────────────────')
verdict = '[GO  — meets all criteria]' if all_pass else '[NO-GO — review failing checks above]'
print(f'  {verdict}')

In [ ]:
# Cell F — Out-of-sample equity curve + drawdown
import matplotlib.pyplot as plt
from src.backtest.runner import run_backtest

split       = int(len(candles) * 0.6)
test_result = run_backtest(candles[split:], params, cfg)
equity      = test_result['equity_curve']

# Drawdown series
peak      = 0.0
drawdowns = []
for e in equity:
    peak = max(peak, e)
    drawdowns.append(e - peak)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7),
                                gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(equity, linewidth=1.2, color='steelblue', label='Net PnL (USDT)')
ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.fill_between(range(len(equity)), equity, 0,
                 where=[e >= 0 for e in equity], alpha=0.15, color='green')
ax1.fill_between(range(len(equity)), equity, 0,
                 where=[e <  0 for e in equity], alpha=0.15, color='red')
ax1.set_title('Out-of-Sample Equity Curve  (40% test window)', fontsize=13)
ax1.set_ylabel('Cumulative PnL (USDT)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.fill_between(range(len(drawdowns)), drawdowns, 0, color='red', alpha=0.4)
ax2.set_ylabel('Drawdown (USDT)')
ax2.set_xlabel('Candle index (5-min bars)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

peak_eq  = max(equity) if equity else 0
final_eq = equity[-1]  if equity else 0
max_dd   = min(drawdowns) if drawdowns else 0
print(f'Peak: ${peak_eq:.4f}  |  Final: ${final_eq:.4f}  |  Max drawdown: ${max_dd:.4f}')